# Phase 6: Model Evaluation



In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
plt.rcParams['font.size'] = 11


## 1. Load Data & Trained Model

In [ ]:
# Load model
model_path = '../models/best_model.pkl'
model = joblib.load(model_path)
print(f"Loaded trained model: {type(model).__name__}")

# Load dataset
df = pd.read_csv('../data/processed/featured_air_quality.csv')
drop_cols = ['city', 'date', 'split', 'aqi_48', 'aqi_72', 'aqi_bucket']
features = [c for c in df.columns if c not in drop_cols and c != 'aqi_24']

test_df = df[df['split'] == 'test'].copy()
X_test = test_df[features].fillna(df[features].median())
y_test = test_df['aqi_24']

preds = model.predict(X_test)
test_df['pred_aqi_24'] = preds
test_df['residual'] = test_df['aqi_24'] - test_df['pred_aqi_24']

rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
print(f"Test Set Performance -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}")

## 2. Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, preds, alpha=0.3, color='#1f77b4', edgecolors='none', s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ideal 1:1 Line')
plt.title('Actual vs Predicted AQI (24h Ahead)', fontsize=14, fontweight='bold')
plt.xlabel('Actual AQI_24')
plt.ylabel('Predicted AQI_24')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/plots/actual_vs_predicted.png', dpi=300)
plt.show()

## 3. Residual Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(preds, test_df['residual'], alpha=0.3, color='#2ca02c', edgecolors='none', s=20)
plt.axhline(0, color='red', linestyle='--', lw=2, label='Zero Error Line')
plt.title('Residual Plot (Errors vs Predicted Values)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted AQI_24')
plt.ylabel('Residual (Actual - Predicted)')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/plots/residual_plot.png', dpi=300)
plt.show()

## 4. Error Distribution Plot

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(test_df['residual'], kde=True, color='#d62728', bins=40)
plt.axvline(test_df['residual'].mean(), color='black', linestyle='--', lw=1.5, label=f"Mean Error: {test_df['residual'].mean():.2f}")
plt.title('Error Distribution (Residual Histogram & KDE)', fontsize=14, fontweight='bold')
plt.xlabel('Prediction Error (Actual - Predicted)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/plots/error_distribution.png', dpi=300)
plt.show()

## 5. Feature Importance

In [ ]:
if hasattr(model, 'feature_importances_'):
    importance = pd.DataFrame({
        'Feature': features,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False).head(15)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance, x='Importance', y='Feature', palette='viridis')
    plt.title('Top 15 Feature Importances', fontsize=14, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.savefig('../outputs/plots/feature_importance.png', dpi=300)
    plt.show()

## 6. SHAP Summary Plot

In [ ]:
explainer = shap.Explainer(model, X_test)
shap_values = explainer(X_test.iloc[:1000])  # sample 1000 for speed

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test.iloc[:1000], show=False)
plt.title('SHAP Summary Plot (Feature Impact on Prediction)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../outputs/plots/shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Forecast Accuracy Across Air Quality Buckets

In [ ]:
# Group actual AQI into standard CPCB buckets
def get_bucket(aqi):
    if aqi <= 50: return 'Good'
    elif aqi <= 100: return 'Satisfactory'
    elif aqi <= 200: return 'Moderate'
    elif aqi <= 300: return 'Poor'
    elif aqi <= 400: return 'Very Poor'
    else: return 'Severe'

test_df['AQI_Bucket_Actual'] = test_df['aqi_24'].apply(get_bucket)
bucket_metrics = test_df.groupby('AQI_Bucket_Actual').apply(
    lambda g: pd.Series({
        'MAE': mean_absolute_error(g['aqi_24'], g['pred_aqi_24']),
        'RMSE': np.sqrt(mean_squared_error(g['aqi_24'], g['pred_aqi_24'])),
        'Count': len(g)
    })
).loc[['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']].dropna()

fig, ax1 = plt.subplots(figsize=(10, 5))
x = np.arange(len(bucket_metrics))
width = 0.35

rects1 = ax1.bar(x - width/2, bucket_metrics['MAE'], width, label='MAE', color='#3182bd')
rects2 = ax1.bar(x + width/2, bucket_metrics['RMSE'], width, label='RMSE', color='#e6550d')

ax1.set_ylabel('Error Metric')
ax1.set_title('Forecast Accuracy Across Air Quality Categories', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(bucket_metrics.index)
ax1.legend()

plt.tight_layout()
plt.savefig('../outputs/plots/forecast_accuracy_chart.png', dpi=300)
plt.show()